In [65]:
import pandas as pd
from pathlib import Path

In [66]:
# --- 1. Path & Connection Configuration ---
BASE_DIR = Path.cwd().parent
DATASET_PATH = BASE_DIR / "data" / "processed" / "nitibench_cleaned_2026-03-17.parquet"

In [67]:
cleaned_df = pd.read_parquet(DATASET_PATH)

In [68]:
cleaned_df.head(2)

,law_name,section_content,section_num
0,พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546,พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มา...,132
1,ประมวลกฎหมายแพ่งและพาณิชย์,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 1598/5\nถ้าผู...,1598/5


In [69]:
for uni in cleaned_df["law_name"].unique():
    print(uni)

พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546
ประมวลกฎหมายแพ่งและพาณิชย์
พระราชบัญญัติหลักประกันทางธุรกิจ พ.ศ. 2558
พระราชบัญญัติการบัญชี พ.ศ. 2543
ประมวลรัษฎากร
พระราชกำหนดการประกอบธุรกิจสินทรัพย์ดิจิทัล พ.ศ. 2561
พระราชบัญญัติหอการค้า พ.ศ. 2509
พระราชบัญญัติหลักทรัพย์และตลาดหลักทรัพย์ พ.ศ. 2535
พระราชบัญญัติกำหนดความผิดเกี่ยวกับห้างหุ้นส่วนจดทะเบียน ห้างหุ้นส่วนจำกัด บริษัทจำกัด สมาคม และมูลนิธิ พ.ศ. 2499
พระราชบัญญัติบริษัทมหาชนจำกัด พ.ศ. 2535
พระราชบัญญัติทรัสต์เพื่อธุรกรรมในตลาดทุน พ.ศ. 2550
พระราชบัญญัติการประกอบธุรกิจของคนต่างด้าว พ.ศ. 2542
พระราชบัญญัติกองทุนสำรองเลี้ยงชีพ พ.ศ. 2530
พระราชบัญญัติวิชาชีพบัญชี พ.ศ. 2547
พระราชบัญญัติสมาคมการค้า พ.ศ. 2509
พระราชกำหนดนิติบุคคลเฉพาะกิจเพื่อการแปลงสินทรัพย์เป็นหลักทรัพย์ พ.ศ. 2540
พระราชบัญญัติทะเบียนพาณิชย์ พ.ศ. 2499
พระราชบัญญัติการส่งเสริมการอนุรักษ์พลังงาน พ.ศ. 2535
พระราชบัญญัติการประกอบกิจการพลังงาน พ.ศ. 2550
พระราชบัญญัติธุรกิจสถาบันการเงิน พ.ศ. 2551
พระราชบัญญัติภาษีเงินได้ปิโตรเลียม พ.ศ. 2514
พระราชบัญญัติทุนรัฐวิสาหกิจ พ.ศ. 2

In [70]:
filter_df = cleaned_df[(cleaned_df["law_name"]=="พระราชบัญญัติหลักทรัพย์และตลาดหลักทรัพย์ พ.ศ. 2535") &
           (cleaned_df["section_num"]=="59")]

In [71]:
filter_df = cleaned_df[(cleaned_df["law_name"]=="พระราชบัญญัติบริษัทมหาชนจำกัด พ.ศ. 2535") &
           (cleaned_df["section_num"]=="50")]

In [72]:
for text in filter_df["section_content"]:
    print(text)

พระราชบัญญัติบริษัทมหาชนจำกัด พ.ศ. 2535 มาตรา 50 หุ้นของบริษัทแต่ละหุ้นต้องมีมูลค่าเท่ากัน


In [73]:
import pandas as pd
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct
from FlagEmbedding import BGEM3FlagModel
import numpy as np

In [74]:

# 2. Initialize Model
MODEL_NAME = "VISAI-AI/nitibench-ccl-human-finetuned-bge-m3"
model = BGEM3FlagModel(
    MODEL_NAME, 
    use_fp16=True, 
    device="cuda" 
)

Fetching 23 files: 100%|██████████| 23/23 [00:00<?, ?it/s]


In [ ]:
rows = []
# QA สามารถกำหนดให้หุ้นของบริษัทแต่ละหุ้นมีมูลค่าไม่เท่ากันได้หรือไม่
rows.append({
    'id': 9999,
    'law_name': "พระราชบัญญัติบริษัทมหาชนจำกัด พ.ศ. 2535",
    'section_num': "50",
    'text': "พระราชบัญญัติบริษัทมหาชนจำกัด พ.ศ. 2535 มาตรา 50 หุ้นของบริษัทแต่ละหุ้นต้องมีมูลค่าเท่ากันและต้องมีมูลค่าไม่ต่ำกว่าห้าบาท"
})


# Manual Ingestion

In [76]:
from qdrant_client.models import SparseVector


for r in rows:
    output = model.encode(r["text"], return_dense=True, return_sparse=True)
    r["dense_vector"] = output['dense_vecs'].tolist()
    
    
    indices: list[int] = []
    values: list[float] = []
    for k, v in output['lexical_weights'].items():
        if v is not None:
            indices.append(int(k))
            values.append(float(v))
    r["sparse_vector"]  = SparseVector(indices=indices, values=values)


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


In [77]:
all_points = []
for row in rows:
    vectors_dict: dict[str, list[float] | SparseVector] = {
        "dense": row['dense_vector'].tolist() if hasattr(row['dense_vector'], 'tolist') else row['dense_vector']
        # dense dtype: list[float32]
    }

    vectors_dict["sparse"] = row['sparse_vector']
    
    all_points.append(
    PointStruct(
                id=int(row['id']),
                vector=vectors_dict,
                payload={
                    "text": row['text'],                      # str
                    "law_name": row['law_name'],              # str
                    "section_num": row['section_num'],        # str
                    "reference_laws": row.get("reference_laws", []),  # list[dict{"law_name": str, "section_num": str}] | list[]
                }
            )
    )

In [78]:
for p in all_points:
    print(p)

id=9999 vector={'dense': [-0.024078369140625, -0.0030765533447265625, -0.0103912353515625, 0.0207672119140625, -0.0224456787109375, 0.0241241455078125, 0.054962158203125, -1.1444091796875e-05, 0.01261138916015625, -0.006572723388671875, -0.01499176025390625, -0.0208282470703125, -0.01593017578125, -0.03680419921875, 0.024261474609375, -0.0240478515625, 0.007572174072265625, -0.01593017578125, -0.0019254684448242188, 0.009307861328125, -0.040740966796875, -0.0158538818359375, 0.013916015625, 0.0225830078125, 0.006999969482421875, 0.00830841064453125, -0.1004638671875, 0.0037593841552734375, 0.00716400146484375, 0.033111572265625, -0.0128173828125, -0.01503753662109375, -0.0016698837280273438, -0.029815673828125, -0.00687408447265625, -0.041015625, -0.030029296875, 0.0110015869140625, -0.01239776611328125, -0.0099029541015625, 0.0175933837890625, -0.0026111602783203125, 0.01044464111328125, -0.0491943359375, 0.007572174072265625, 0.0025081634521484375, -0.0266571044921875, -0.02890014648

In [79]:
QDRANT_URL = "http://localhost:6333" 
COLLECTION_NAME = "thai_laws_collection"

# --- 2. Initialize Client & Model ---
client = QdrantClient(url=QDRANT_URL, timeout=60) # เชื่อมต่อผ่าน URL

In [80]:
# --- ขั้นตอนที่ 2: เริ่มส่งข้อมูลเข้า Qdrant (Ingestion) ---
# มั่นใจได้ว่าข้อมูลทุกอย่างพร้อมแล้วใน all_points
# ตัวอย่างการทำ Batching แบบง่าย
batch_size = 100
for i in range(0, len(all_points), batch_size):
    batch = all_points[i : i + batch_size]
    client.upsert(
        collection_name=COLLECTION_NAME,
        points=batch
    )
    print(f"Uploaded {i + len(batch)} points...")

Uploaded 1 points...
